# Rubric review on a DYNAMO VM — FPPG Project Deliverables Checklist

Applies the **C – Modeling Review** checklist to **every HEC-RAS plan and HEC-HMS simulation run found in the models you provide**.
The checklist workbook is a *template*: its example model names are ignored. Nothing is matched by name.

Each model gets the three checklist tests (runs without errors, Scope of Work events, project area) and the notebook writes:

* an HTML **report** organised by theme (hydrologic / hydraulic), with findings, evidence and maps, and
* a copy of the checklist workbook with one block per model found and *TWDB PM Draft Review* (Included Y/N + Comment) filled in.

**Inputs:** the checklist `.xlsx` (wording and layout); the model submission `.zip` (download from SharePoint first — the notebook cannot sign in to SharePoint) or a folder already on the VM; the Scope of Work event list; optionally the project-area shapefile.
**Outputs:** `outputs/rubric-review/<date>-<host>/` (git-ignored).
**Software:** the VM's installed HEC-RAS / HEC-HMS (versions are recorded in the report). Run `pip install -e ".[rubric]"` first.

**Result meanings:** `Pass`/`Fail` are automated findings. A blank *Included* cell with a comment means a human must decide. Nothing is written back to SharePoint or to the VM's original files; zips are extracted into a scratch folder.

**Running on several VMs at once:** in `winrm` mode `HOSTS` can list every DYNAMO VM (`dynamo.tacc.utexas.edu` and `dynamo02`–`dynamo08`). The submission is staged on each VM and the plans/runs are shared out: each VM takes the next untested model as soon as it is free, so a review that would take hours on one VM finishes in roughly that time divided by the number of VMs. Findings are merged into one report that names the VM that tested each model.

**Prerequisites:** `winrm` mode — `pip install -e ".[rubric]"` on your own machine, network access to port 5985, and an account that is an Administrator or in *Remote Management Users* on the VM (put `username`/`password` in the git-ignored `.env`). `local` mode — Python 3.10+ and this repo on the VM. The project-area map needs the model files readable by Python, so it works in `local` mode only.

In [1]:
import sys
from datetime import date
from pathlib import Path

REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO))  # use the checkout when the package is not pip-installed

from rubric_review import (DYNAMO_HOSTS, Fleet, NotebookProgress, ReviewOptions, build_review_workbook, run_review,
                           run_review_fleet, write_html_report)
from rubric_review.local import LocalSession

## What each rubric item tests

| Rubric item | Test | You must provide |
|---|---|---|
| **Model runs with no errors** | Plan's geometry + flow files exist. With `RUN_MODELS=True`: compute the plan/run, then wait for it (progress is printed every few seconds, no time limit) and check the exit code, fresh results file, and error lines in the compute messages. | Nothing (HEC installs are detected). |
| **All events in the Scope of Work are represented** | HMS: every expected event has a simulation run. RAS: every expected event is named in the plan title, flow title or profile names (`10-yr`, `100 year`, and `1%` forms all match). The report shows a model × event matrix. | `EXPECTED_EVENTS` — the event list from the Scope of Work. |
| **Project Area in the Grant Agreement is represented** | RAS: the project-area shapefile is referenced in the `.rasmap` **and** ≥ `MIN_COVERAGE` of its area lies inside the model footprint (read from the geometry HDF). A map figure is saved per model. HMS: shapefile reference only (no geographic check yet). | `PROJECT_AREA_SHP` — the Grant Agreement shapefile, on the VM. Footprint needs `conda install geopandas h5py matplotlib`. |

Anything that cannot be decided automatically is left blank for the reviewer with the evidence in the comment.

## 1. Parameters

In [2]:
MODE = "winrm"                         # "winrm": run from your own machine over PowerShell remoting (port 5985; verified on dynamo03).
                                       # "local": notebook runs ON the Windows VM (use after you RDP in).
                                       # "ssh": SSH login (port 22 is closed from outside, so not currently usable).
HOSTS = DYNAMO_HOSTS                   # winrm: VMs to spread the work over; DYNAMO_HOSTS = dynamo.tacc.utexas.edu + dynamo02..08.
                                       # Use a single-item list, e.g. ["dynamo03.tacc.utexas.edu"], to run on one VM. local/ssh use HOSTS[0].
USERNAME = None                        # winrm/ssh: None = read `username` from REPO/.env (e.g. production\dynamo), else prompt

RUBRIC = REPO / "docs/rubrics/FPPG_Project_Deliverables Checklist_FINAL.xlsx"

# Provide ONE of these:
LOCAL_MODELS_ZIP = Path.home() / "Downloads/OneDrive_1_10-27-2026.zip"              # e.g. Path.home() / "Downloads/Test1_10_26_2026.zip"  (extracted into a scratch folder on the VM)
VM_MODELS_ROOT = None                  # e.g. r"C:\Users\dynamo\Downloads\Test1_10_26_2026\Test1_10_26_2026" (already extracted)

RUN_MODELS = True                     # False: locate projects + check dependencies. True: also compute every plan/run (slow).
RAS_VERSION = "6.6"                    # preferred HEC-RAS version; falls back to the highest installed
HMS_VERSION = None                     # None = highest installed HEC-HMS
EXPECTED_EVENTS = []   # EDIT to match the Scope of Work; [] = evidence only
PROJECT_AREA_SHP = None                # e.g. Path(r"C:\Users\dynamo\Downloads\GrantArea\ProjectArea.shp"); None = map-layer check only
MIN_COVERAGE = 0.95                    # share of the project area that must fall inside the model footprint
POLL_SECONDS = 30                      # how often a running compute is checked and a status line printed (no time limit; interrupt the cell to stop it)

import time
_t0 = time.monotonic()
def say(message):
    # timestamped status line, printed immediately
    print(f"[{time.monotonic() - _t0:6.0f}s] {message}", flush=True)

HOST = HOSTS[0]
RUN_LABEL = HOST.split(".")[0] if len(HOSTS) == 1 else f"{len(HOSTS)}-vms"
OUT_DIR = REPO / "outputs/rubric-review" / f"{date.today():%Y-%m-%d}-{RUN_LABEL}"
print("Checklist template:", RUBRIC.name)

Checklist template: FPPG_Project_Deliverables Checklist_FINAL.xlsx


## 2. Connect

`winrm` logs in to the VMs in `HOSTS` one after another with the account in `.env` (or a hidden prompt), one attempt each and nothing held on disk. The first VM must succeed before any other is tried: a wrong password tried on eight VMs is eight failed logins on the same account, which locks it. A later VM that cannot be reached is skipped and the review runs on the rest. `local` needs no login: your RDP session is the authentication. Run this cell once; if it fails, check the account before re-running.


In [3]:
fleet = None  # winrm: all connected VMs; `session` is always the first one (used for local/ssh and single-VM paths)
if MODE == "local":
    session = LocalSession()
    print("Running locally on", session.run_script("inventory_software.ps1")["computer"])
elif MODE == "winrm":
    # ONE login attempt per VM, no retries (repeated failures lock the account). Needs Administrator or
    # "Remote Management Users" on the VMs. Credentials come from the git-ignored .env, else hidden prompts.
    from getpass import getpass
    from rubric_review.winrm import load_env
    env = load_env(REPO / ".env") if (REPO / ".env").exists() else {}
    user = USERNAME or env.get("username") or input(r"VM username (e.g. production\dynamo): ")
    password = env.get("password") if not USERNAME else None
    if password is None:
        password = getpass(f"{user} password: ")
    fleet = Fleet.connect_winrm(HOSTS, user, password, progress=say)
    session = fleet.sessions[fleet.hosts[0]]
    print(f"Connected to {len(fleet.hosts)} of {len(HOSTS)} VM(s) as {user}: {', '.join(h.split('.')[0] for h in fleet.hosts)}")
elif MODE == "ssh":
    from rubric_review.remote import VMSession
    session = VMSession(HOST, USERNAME or input("SSH username: ")).connect()
    print("Connected to", HOST)
else:
    raise ValueError("MODE must be 'winrm', 'local' or 'ssh'")


[     3s] [dynamo] connected
[     4s] [dynamo02] connected
[     4s] [dynamo03] connected
[     5s] [dynamo04] connected
[     6s] [dynamo05] connected
[     7s] [dynamo06] connected
[     8s] [dynamo07] connected
[     9s] [dynamo08] connected
Connected to 8 of 8 VM(s) as production\dynamo: dynamo, dynamo02, dynamo03, dynamo04, dynamo05, dynamo06, dynamo07, dynamo08


## 3. Stage the models on the VM(s)

With several VMs the zip is uploaded to all of them at the same time (each upload runs at roughly 1 MB/s, so a large submission takes a while) and extracted to the same path on each; the cell checks that every VM ended up with the same files.


In [4]:
if (LOCAL_MODELS_ZIP is None) == (VM_MODELS_ROOT is None):
    raise ValueError("Set exactly one of LOCAL_MODELS_ZIP or VM_MODELS_ROOT")

if LOCAL_MODELS_ZIP is not None:
    dest = f"rubric-review\\models-{date.today():%Y%m%d}"
    if MODE == "local":
        say("Extracting the zip")
        staged = session.run_script("stage_models.ps1", timeout=1800, Zip=str(LOCAL_MODELS_ZIP), Dest=str(Path.home() / "rubric-review" / f"models-{date.today():%Y%m%d}"))
        models_root = staged["root"]
        say(f"Extracted {staged['fileCount']} files to {models_root}")
    elif fleet is not None:
        say(f"Staging {Path(LOCAL_MODELS_ZIP).name} ({Path(LOCAL_MODELS_ZIP).stat().st_size / 1e6:.0f} MB) on {len(fleet.hosts)} VM(s); this can take several minutes")
        models_root = fleet.stage(LOCAL_MODELS_ZIP, dest, progress=say)
        say(f"Every VM has the models at {models_root}")
    else:
        say(f"Uploading {Path(LOCAL_MODELS_ZIP).name} ({Path(LOCAL_MODELS_ZIP).stat().st_size / 1e6:.0f} MB) to {HOST}; this can take several minutes")
        zip_on_vm = session.upload(LOCAL_MODELS_ZIP, "models.zip")
        say("Upload complete; extracting the zip on the VM")
        staged = session.run_script("stage_models.ps1", timeout=1800, Zip=zip_on_vm, Dest=dest)
        models_root = staged["root"]
        say(f"Extracted {staged['fileCount']} files to {models_root}")
else:
    models_root = VM_MODELS_ROOT
    print("Using models already on the VM(s):", models_root, "(must be the same path on every VM)")


[     9s] Staging OneDrive_1_10-27-2026.zip (1323 MB) on 8 VM(s); this can take several minutes
[     9s] [dynamo] uploading OneDrive_1_10-27-2026.zip (1323 MB)
[     9s] [dynamo03] uploading OneDrive_1_10-27-2026.zip (1323 MB)[     9s] [dynamo02] uploading OneDrive_1_10-27-2026.zip (1323 MB)

[     9s] [dynamo04] uploading OneDrive_1_10-27-2026.zip (1323 MB)
[     9s] [dynamo05] uploading OneDrive_1_10-27-2026.zip (1323 MB)
[     9s] [dynamo06] uploading OneDrive_1_10-27-2026.zip (1323 MB)
[     9s] [dynamo07] uploading OneDrive_1_10-27-2026.zip (1323 MB)
[     9s] [dynamo08] uploading OneDrive_1_10-27-2026.zip (1323 MB)
[  1995s] [dynamo08] upload done in 1986s; extracting
[  1996s] [dynamo06] upload done in 1986s; extracting
[  1998s] [dynamo08] extracted 125 files to C:\Users\dynamo\rubric-review\models-20261103
[  1999s] [dynamo04] upload done in 1989s; extracting
[  1999s] [dynamo06] extracted 125 files to C:\Users\dynamo\rubric-review\models-20261103
[  1999s] [dynamo03] upload 

## 4. Run the rubric

Progress is one table that updates in place: an overall **N of M models complete** bar, one row per VM showing the model it is computing, the stage (preprocessing geometry, unsteady flow, writing results) and elapsed time, and a short list of recent events. Interrupting the cell stops the compute on every VM. Per-poll details are still written to `OUT_DIR/logs`.


In [ ]:
geo = None
if PROJECT_AREA_SHP is not None:
    if MODE != "local":
        print("Footprint check needs local mode (reads model HDF files directly); falling back to the map-layer check.")
    else:
        from rubric_review.geo import LocalGeo
        geo = LocalGeo(PROJECT_AREA_SHP, OUT_DIR / "figures", MIN_COVERAGE)

board = NotebookProgress()  # one table that updates in place: "N of M models complete" + a row per VM (model, stage, elapsed)
options = ReviewOptions(
    models_root=models_root, run_models=RUN_MODELS, ras_version=RAS_VERSION, hms_version=HMS_VERSION,
    project_area_shapefile=Path(PROJECT_AREA_SHP).name if PROJECT_AREA_SHP else None,
    expected_events=EXPECTED_EVENTS, poll_seconds=POLL_SECONDS, log_dir=OUT_DIR / "logs",
    progress=board.say, status=board.status,
)
if fleet is not None and len(fleet.hosts) > 1:
    outcome = run_review_fleet(fleet.calls, options, geo=geo)
else:
    outcome = run_review(session.run_script, options, geo=geo)
print(outcome.environment)

## 5. Write the report

In [6]:
xlsx = build_review_workbook(RUBRIC, OUT_DIR / RUBRIC.name.replace(".xlsx", "_REVIEWED.xlsx"), outcome.findings)
html = write_html_report(outcome, OUT_DIR / "report.html", "FPPG Project Deliverables — Modeling Review",
                         ", ".join(h.split(".")[0] for h in (fleet.hosts if fleet is not None else [HOST])))
print("Workbook:", xlsx)
print("Report:  ", html)

from IPython.display import Image, display
print(f"\n{len(outcome.findings)} plans/runs found and tested:")
for f in outcome.findings:
    where = f"  [{f.host.split('.')[0]}]" if f.host else ""
    print(f"  {f.tool.upper():3} {f.project} | {f.unit}  ->  " + "  ".join(f"{k}={s or 'REVIEW'}" for k, (s, _c, _f) in f.checks.items()) + where)
    for _s, _c, fig in f.checks.values():
        if fig:
            display(Image(fig, width=420))
print("\nProjects found under", models_root)
for line in outcome.inventory["hms"] + outcome.inventory["ras"]:
    print("  ", line)
if not (outcome.inventory["hms"] or outcome.inventory["ras"]):
    print("   NOTHING FOUND - models_root has no HEC-RAS (.prj) or HEC-HMS (.hms) projects; check the path.")
print("\nOpen the report in a browser:", html)
print("Per-poll run logs (what each tool wrote while computing):", OUT_DIR / "logs")

Workbook: /Users/wmobley/Documents/Github/twdb-flood-model-automation/outputs/rubric-review/2026-11-03-8-vms/FPPG_Project_Deliverables Checklist_FINAL_REVIEWED.xlsx
Report:   /Users/wmobley/Documents/Github/twdb-flood-model-automation/outputs/rubric-review/2026-11-03-8-vms/report.html

14 plans/runs found and tested:
  RAS SwissAlpine_2D | Ex_1%  ->  runs=Y  events=REVIEW  area=Y  [dynamo]
  RAS SwissAlpine_2D | Alt1_Detention_01%  ->  runs=Y  events=REVIEW  area=Y  [dynamo07]
  RAS SwissAlpine_2D | Alt1_Detention_10%  ->  runs=Y  events=REVIEW  area=Y  [dynamo05]
  RAS SwissAlpine_2D | EX_10%  ->  runs=Y  events=REVIEW  area=Y  [dynamo06]
  RAS SwissAlpine_2D | Alt2_Diversion_01%  ->  runs=Y  events=REVIEW  area=Y  [dynamo08]
  RAS SwissAlpine_2D | Ex_2%  ->  runs=Y  events=REVIEW  area=Y  [dynamo04]
  RAS SwissAlpine_2D | Ex_0.2%  ->  runs=Y  events=REVIEW  area=Y  [dynamo03]
  RAS SwissAlpine_2D | Alt2_Diversion_10%  ->  runs=Y  events=REVIEW  area=Y  [dynamo02]
  RAS SwissAlpine_2D

In [7]:
(fleet or session).close()  # drops every connection and the in-memory password
